# Whisper Fine-Tuning — Multi-Dataset Duration Sweep

Loops over progressively larger nested training manifests (e.g. 1 h → 10 h).
For each dataset the notebook:
1. Reloads the base Whisper model fresh
2. Prepares train/validation splits from the duration manifest
3. Runs fine-tuning
4. Pushes the fine-tuned checkpoint to the Hub
5. Evaluates on the shared held-out test set
6. Saves per-run prediction CSV and appends a summary row to a results table

## 1. Python Setup

In [2]:
ENV = "local"  # options: "local", "colab"

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from huggingface_hub import login

# ── Project path ──────────────────────────────────────────────────────────────
if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_SRC = Path("/content/drive/MyDrive/chichewa-asr/src")
else:
    PROJECT_SRC = Path().cwd().parents[0]

sys.path.append(str(PROJECT_SRC))

from src.train.train_whisper import load_config
from src.train.whisper_duration_experiment import (
    load_model_and_processor,
    prepare_train_dataset,
    prepare_test_dataset,
    run_training,
    run_evaluation,
)

## 2. Paths and Global Configuration

In [4]:
# ==========================================
# 1. BASE WORKING DIRECTORY AND PATHS
# ==========================================
# Base directory of the project (one level up from src/)
DIR_BASE   = Path.cwd().parents[0]
DIR_DATA   = DIR_BASE / "data"

# Hold out test set 
DIR_TEST               = DIR_DATA / "test"
FILE_MANIFEST_TEST     = DIR_TEST / "metadata.csv"

# Audio files for the dev set and the dev set with nested duration
DIR_DEV                     = DIR_DATA / "dev"
DIR_DEV_NESTED_DURATION     = DIR_DATA / "dev_nested_duration"

# Hyperparameter config file path
FILE_CONFIG = DIR_BASE / "configs" / "whisper_params.yaml"

# Outputs directory where we keep the results of the experiments
DIR_OUTPUTS = DIR_BASE / "outputs"
DIR_RESULTS = DIR_OUTPUTS / "duration_exp_whisper_small"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. HARDWARE SETUP
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


In [5]:
# ==========================================
# 3. LOGIN TO HUGGINGFACE HUB
# ==========================================
if ENV == "colab":
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
else:
    load_dotenv()
    login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 3. Experiment Functions

All orchestration lives in `src/train/`. No functions needed here.

| Module | Responsibility |
|--------|----------------|
| `base_duration_experiment.py` | Config patching, evaluation loop, Hub upload, result saving |
| `whisper_duration_experiment.py` | Model loading, dataset prep, Seq2Seq trainer, decoding |

In [ ]:
# All experiment functions are imported above from:
#   src/train/whisper_duration_experiment.py  (Whisper-specific)
#   src/train/base_duration_experiment.py     (generic scaffold)
#
# To switch to MMS, change the import in cell 3 to:
#   from src.train.mms_duration_experiment import prepare_test_dataset, run_experiment


## 4. Load Base Config and Held-Out Test Set

These two objects are created once and shared across every duration run.
The test set is preprocessed with a temporary processor (base model weights);
audio features are model-agnostic so this is correct for all runs.

In [6]:
# Load base config once
base_config = load_config(FILE_CONFIG)
print("Config loaded:", FILE_CONFIG)

# Pre-process the held-out test set once — reused across all runs.
# prepare_test_dataset builds its own processor internally from base_config.
dataset_test = prepare_test_dataset(
    FILE_MANIFEST_TEST,
    audio_dir=DIR_TEST,
    base_config=base_config,
    audio_fname_col="audio_filename",
    duration_col="duration_seconds",
)

print(f"\nHeld-out test set ready: {len(dataset_test):,} utterances")


Config loaded: /Users/dmatekenya/git-repos/chichewa-asr/configs/whisper_params.yaml
  Loading test data: /Users/dmatekenya/git-repos/chichewa-asr/data/test/metadata.csv
Total duration : 1.68 hrs  (573 utterances)
  all_data   :   573 utterances  |  1.68 hrs (100.0%)


Map (num_proc=1): 100%|██████████| 573/573 [00:12<00:00, 44.88 examples/s]


Held-out test set ready: 573 utterances


## 5. Define Duration Datasets 

Provide paths to all target nested duration manifest files. 

In [9]:
# ================================
# BUILD DURATION_DATASETS 
# ===============================

DURATION_DATASETS = {
    f"{h}h": DIR_DEV_NESTED_DURATION / f"train_{h}h.csv"
    for h in range(1, 3)
}

# Sanity-check that all manifests exist before launching long runs
missing = [str(p) for p in DURATION_DATASETS.values() if not Path(p).exists()]
if missing:
    print("WARNING — manifest files not found:")
    for m in missing:
        print(f"  {m}")
else:
    print(f"All {len(DURATION_DATASETS)} manifest files found. Ready to run.")

All 2 manifest files found. Ready to run.


## 6. Run Fine-tuning 

Each iteration: loads a fresh base model → trains → (optionally) pushes to Hub
→ evaluates on the shared test set → saves predictions CSV.

In [10]:
base_config

{'experiment': {'name': 'whisper_large_v3_chichewa_base',
  'output_dir': 'outputs/whisper_large_v3_chichewa_base',
  'hub_model_id': 'dmatekenya/whisper-large-v3-chichewa-base',
  'seed': 42},
 'model': {'model_name_or_path': 'openai/whisper-small',
  'language': 'swahili',
  'task': 'transcribe'},
 'training': {'per_device_train_batch_size': 2,
  'gradient_accumulation_steps': 1,
  'learning_rate': 1e-05,
  'warmup_steps': 5,
  'max_steps': 20,
  'gradient_checkpointing': False,
  'fp16': False,
  'predict_with_generate': True,
  'generation_max_length': 225,
  'save_strategy': 'steps',
  'save_steps': 10,
  'logging_steps': 5},
 'evaluation': {'eval_strategy': 'steps',
  'eval_steps': 10,
  'load_best_model_at_end': True,
  'metric_for_best_model': 'wer',
  'greater_is_better': False},
 'hub': {'push_to_hub': False, 'report_to': 'none'}}

In [ ]:
summary = []

for duration_label, manifest_path in DURATION_DATASETS.items():
    if not manifest_path.exists():
        print(f"Skipping {duration_label} — manifest not found.")
        continue

    print(f"\n{'='*60}\n  EXPERIMENT: {duration_label}\n{'='*60}")

    # ==========================================
    # 1. PER-RUN CONFIG
    # ==========================================
    run_config = build_run_config(base_config, duration_label)

    # ==========================================
    # 2. LOAD MODEL AND PROCESSOR
    # ==========================================
    model, processor = load_model_and_processor(run_config)

    # ==========================================
    # 3. PREPARE TRAINING DATA
    # ==========================================
    dataset_train = prepare_train_dataset(manifest_path, DIR_DEV, processor)

    # ==========================================
    # 4. TRAIN
    # ==========================================
    trainer = run_training(model, processor, dataset_train, run_config)

    # ==========================================
    # 5. PUSH TO HUB
    # ==========================================
    push_to_hub(trainer, run_config)

    # ==========================================
    # 6. EVALUATE ON HELD-OUT TEST SET
    # ==========================================
    row = run_evaluation(model, processor, dataset_test, duration_label, DIR_RESULTS)
    summary.append({"duration": duration_label, **{k: v for k, v in row.items() if k != "predictions"}})

    # Save rolling summary after every run so partial results survive a crash
    pd.DataFrame(summary).to_csv(DIR_RESULTS / "sweep_summary.csv", index=False)

## 7. Results Summary

In [ ]:
df_summary = pd.read_csv(DIR_RESULTS / "sweep_summary.csv")
display(df_summary.style.format({"wer": "{:.2f}%", "cer": "{:.2f}%"}))